# Data Cleaning

In this notebook, we will begin cleaning up our IMDb and Oscar datasets in order to create visualizations that will help invesitgating our narrative.

## Preprocessing

We only need to import Pandas for reading in .csv files, creating dataframes, and to mainly help with cleaning the data. The AST library is for abstract syntax trees, which can help with changing datatypes from strings to lists when using Regex is not possible (this problem will be shown later below).

In [ ]:
# libraries needed
import pandas as pd
import ast

### Initializing helper functions

We created a file with functions that help read and write data after filtering, which we initialize at the top of our program with Jupyter Magics.

In [ ]:
%run helpers.ipynb

In [ ]:
# read in imdb dataset
raw_imdb_df = pd.read_csv("imdb_data/imdb_full_data.csv")
raw_imdb_df.head(10)

In [ ]:
# read in oscar dataset
raw_oscar_df = pd.read_csv("oscar_data/oscar_full_data.csv", sep="\t")
raw_oscar_df.head(10)

In [ ]:
# number of rows and columns for each dataset
print("Raw IMDb dataset shape: ", raw_imdb_df.shape)
print("Raw Oscar dataset shape: ", raw_oscar_df.shape)

## Filtering Columns - IMDb data

We want to begin dropping any columns with irrelevant information that our IMDb dataset provides us. These include MPA (Motion Picture Association), film duration, film description, IMDd link, and filming locations. These columns are fine to drop since they won't be needed for our investigation since they are only basic and additional film details. In addition, the 'awards_content' column was dropped since all the row values are empty.

The next step is to drop any empty row values in the IMDb rating or user votes columns. These two fields will be helpful for measuring audience interest.

Finally, we can go ahead and sort all of the films in the IMDb dataset by release date, starting with the earliest (from 1920 to 2025).

In [ ]:
# keep all relevant rows in imdb dataset
imdb_df = raw_imdb_df.drop(columns=['mpa', 'duration', 'description', 'movie_link', 'filming_locations', 'awards_content'])

# filter out NaNs in rating or votes columns
imdb_df = imdb_df.dropna(subset=['rating', 'votes'], how='any')

# sort release date by ascending order (convert column from string to datetime)
imdb_df = imdb_df.assign(release_date=pd.to_datetime(imdb_df['release_date'])).sort_values(by='release_date').reset_index(drop=True)

imdb_df.head()


We've managed to filter out around 4,000 rows and 6 columns in the IMDb dataset so far, as seen by the shape of the dataframe.

In [ ]:
# display shape of cleaned imdb dataset
print("Cleaned IMDb dataset shape: ", imdb_df.shape)

## Filtering Columns - Oscar data

For the Oscar nominee dataset, we will also filter out any irrelevant columns. The fields `NomId` and `NomineeIds` can be dropped because these IDs do not appear in the IMDb dataset and would not help with comparison across both datasets. Instead, we want to join the two datasets together later on using `FilmId` (*called `id` in the IMDb dataset*), so we must filter out any empty row values in that column.

`Detail` gives information about a character's name in a film, `Note` is additional information prodived about the award, and `Citation` is the official text of the award statement. These columns don't have any use in our investigation and mostly contain empty row values, so they can be dropped. Only 40 entries in the Oscar dataset had data for `MultifilmNomination`, so we chose to drop it.

The field `Year` will be converted from strings to floats. One thing to note is that the first 6 years of the Oscar Awards spanned 12+ months, which is why the periods from 1927-1933 were displayed as 'YYYY/YY' like '1927/28'. The Oscar's did not become a calendar year event until 1934, therefore the logic in the code below for converting these years will reflect that.

Finally, we want to focus this investigation exclusively on feature films, so short film formats or documentaries will not be joined. Therefore, we decided to drop any rows in the dataset that contained the words 'Documentary' or 'Short' in their respective award categories.

The Oscars also contain nomination categories that are not specific to a certain film (honorary or special awards), so simply dropping rows where `FilmId` is empty filters them out of the dataset.

In [ ]:
# keep all relevant rows in oscar dataset
oscar_df = raw_oscar_df.drop(columns=['NomId', 'NomineeIds', 'Detail', 'Note', 'Citation', 'MultifilmNomination'])

# filter out NaNs in FilmId
oscar_df = oscar_df.dropna(subset=['FilmId'])

# filter out irrelevant nomination categories in oscar dataset 
# documentary, short-form, special-award, or legacy categories
oscar_df = oscar_df[~oscar_df['Category'].str.lower().str.contains("documentary|short", na=False)]

# convert years from strings to ints
# Note: the Oscar's did not become a calendar year event until 1934
x = oscar_df['Year'].astype(str)

oscar_df['Year'] = (
    oscar_df['Year']
    .astype(str)
    .str[:4] # take first 4 digits of year
    .astype(int)
    .add(x.str.contains('/').astype(int)) # add one to year if '/' is present
)

oscar_df.head()

We've managed to filter out around 3,000 rows and 6 columns in the Oscar dataset so far, as seen by the shape of the dataframe.

In [ ]:
# display shape of cleaned oscar dataset
print("Cleaned Oscar dataset shape: ", oscar_df.shape)

## Converting datatypes - IMDb data

Nearly all of the data in the IMDb dataset are strings, so we went ahead and assigned them to their appropriate datatypes.

The column `meta_score` was renamed to be indexed without the grave accent, and the row values were converted into integers. The `Int64` datatype was used because it's a nullable integer and can handle the empty row values in the dataset. The row values in the `rating` column were converted to floats.

For the `votes` column, many user votes in the triple and quadruple digits showed up as strings like "9K" for 9,000 votes and "1.1M" for 1,100,000 votes. Therefore, we used Regex to replace these characters with numerical values and make sure that the row values were converted to integers. The `Int64` datatype is used again to handle empty row values.

There are a number of columns that contain data about a film's box office performance, such as `budget` or `gross_worldwide`. Since these values contain characters in them (ex: "$300,000 estimated"), we used Regex to filter out everything except for digits in row values for the `budget`, `opening_weekend_gross`, `gross_worldwide`, and `gross_us_canada` columns.

Lastly, we converted any columns with row values that contained multiple elements, such as `genre` containing `[Drama, Mystery]`. This appears like a list, but is actually a string, so we leveraged the AST (Abstract Syntax Tree) library to convert these strings to lists. We also applied this method to the `writers`, `directors`, `stars`, `countries_origin`, `production_companies`, and `languages` columns. One limitation with this task was that some actors, writers, and directors could have names with apostrophes in them, which the AST library also converts properly.

In [ ]:
# convert meta_score/ratings in imdb dataset from strings to ints/floats
imdb_df = imdb_df.rename(columns={'méta_score': 'meta_score'})
imdb_df['meta_score'] = pd.to_numeric(imdb_df['meta_score']).astype('Int64') # nullable integer type (use since some meta_scores are NaN)
imdb_df['rating'] = pd.to_numeric(imdb_df['rating'])

# convert votes in imdb dataset from strings (ex: 9K, 1.1M) to ints
imdb_df['votes'] = (
    imdb_df['votes']
    .astype(str)
    .str.replace(r'K$', 'e3', regex=True)
    .str.replace(r'M$', 'e6', regex=True)
    .apply(pd.to_numeric) # convert to float to translate e3/e6 into digits
    .astype(int) # use instead of Int64 since NaNs have been dropped already
)

# convert budget/gross numbers in imdb dataset from strings (ex: $300,000 estimated) to ints
# imdb_df[['budget', 'opening_weekend_gross', 'gross_worldwide', 'gross_us_canada']] = (
#     imdb_df[['budget', 'opening_weekend_gross', 'gross_worldwide', 'gross_us_canada']]
#     .replace(r'[^\d]', '', regex=True) # removes everything except for digits
#     .apply(pd.to_numeric) # convert to float to handle NaNs
#     .astype("Int64") # use nullable integer type since some values are NaN
# )

# clean up text formatting for columns in imdb database (convert from strings to lists)
imdb_df[['writers', 'directors', 'stars', 'countries_origin', 'production_companies', 'genres', 'languages']] = (
    imdb_df[['writers', 'directors', 'stars', 'countries_origin', 'production_companies', 'genres', 'languages']]
    .apply(
        lambda col: col.map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    )
)

# display imdb dataframe after changes
imdb_df.head(10)

Note: We did not make any changes to the data types in the Oscar dataset, as most of these data are strings, which is appropriate for what they are representing (category, film, nominees, etc.). There is a column containing the year that is formatted in back to back year format rather than a datetime format (ex: 1927-28), but we decided to leave them untouched for now.

### Writing to CSV

To reuse our filtered data in other notebooks for joining and visualizing, we use our helper function to write the data frames to CSV files that can be read in later

In [ ]:
write_dfs(oscar_df, imdb_df)

In [ ]:
sorted(
    raw_imdb_df["budget"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.extract(r"^([^\w\d\s]|[A-Z]{3})", expand=False)
    .dropna()
    .unique()
)